<a href="https://colab.research.google.com/github/boss-defender/FineTune/blob/main/SmartFineTuner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# 🎯 ALL-IN-ONE UNSLOTH FINE-TUNING PIPELINE (STEPS 1 - 9)
# ==============================================================================

# ------------------------------------------------------------------------------
# STEP 1: GPU & HARDWARE CHECK ⚡
# ------------------------------------------------------------------------------
import torch
import sys

if not torch.cuda.is_available():
    raise RuntimeError("❌ No GPU detected! Please switch Colab runtime to GPU (T4 / V100 / A100) in Runtime -> Change runtime type.")

gpu_name = torch.cuda.get_device_name(0)
print(f"✅ GPU DETECTED: {gpu_name}! Hardware verified.")


# ------------------------------------------------------------------------------
# STEP 2: MOUNT GOOGLE DRIVE 📂
# ------------------------------------------------------------------------------
from google.colab import drive
import os

drive.mount('/content/drive')
print("✅ Google Drive connected!")


# ------------------------------------------------------------------------------
# STEP 3: INSTALL DEPENDENCIES SILENTLY 📦
# ------------------------------------------------------------------------------
print("\n🔄 Installing Unsloth & required libraries...")
!pip install --quiet --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --quiet --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install --quiet "tokenizers<=0.23.2" "datasets" "huggingface_hub" "hf_transfer"
print("✅ Packages installed successfully!")


# ------------------------------------------------------------------------------
# STEP 4: DEFINE CONFIGURATION & HYPERPARAMETERS ⚙️
# ------------------------------------------------------------------------------
MODEL_NAME = "Qwen/Qwen2.5-1.5B"  #@param {type:"string"}

DATASET_NAME = "bespokelabs/Bespoke-Stratos-35k" #@param {type:"string"}
DATASET_REVISION = "main"
DATASET_SPLIT = "train"

MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True

# Training Hyperparameters
LEARNING_RATE = 2e-4
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
MAX_STEPS = 60
WARMUP_STEPS = 5
SAVE_STEPS = 20


# ------------------------------------------------------------------------------
# STEP 5: CONFIG HASHING & DYNAMIC PATHS 🔒
# ------------------------------------------------------------------------------
import hashlib
import json
import re

config_dict = {
    "model_name": MODEL_NAME,
    "dataset_name": DATASET_NAME,
    "max_seq_length": MAX_SEQ_LENGTH,
    "load_in_4bit": LOAD_IN_4BIT,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "grad_accum": GRADIENT_ACCUMULATION_STEPS
}

config_hash = hashlib.sha256(json.dumps(config_dict, sort_keys=True).encode('utf-8')).hexdigest()[:8]

clean_model = re.sub(r'[^a-zA-Z0-9_\-]', '_', MODEL_NAME.split('/')[-1])
clean_dataset = re.sub(r'[^a-zA-Z0-9_\-]', '_', DATASET_NAME.split('/')[-1])

RUN_FOLDER_NAME = f"{clean_model}__{clean_dataset}__{config_hash}"

# Saves tiny LoRA checkpoints to Drive (~150MB) and full merged model in local Colab disk
DRIVE_CHECKPOINT_DIR = f"/content/drive/MyDrive/unsloth_checkpoints/{RUN_FOLDER_NAME}"
LOCAL_FINAL_SAVE_DIR = f"/content/merged_{RUN_FOLDER_NAME}"

print(f"\n📂 UNIQUE RUN DIRECTORY: {RUN_FOLDER_NAME}")
print(f"🔒 SHA-256 Config Hash: {config_hash}")


# ------------------------------------------------------------------------------
# STEP 6: LOAD BASE MODEL & TOKENIZER 📥
# ------------------------------------------------------------------------------
from unsloth import FastLanguageModel

print(f"\n📥 Loading Base Model: {MODEL_NAME}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = LOAD_IN_4BIT,
    revision = MODEL_REVISION
)
print("✅ Base Model & Tokenizer loaded!")


# ------------------------------------------------------------------------------
# STEP 7: LOAD & FORMAT DATASET 📑
# ------------------------------------------------------------------------------
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template, standardize_sharegpt

print(f"\n📥 Loading Dataset: {DATASET_NAME}...")
dataset = load_dataset(DATASET_NAME, split = DATASET_SPLIT, revision = DATASET_REVISION)

# Apply Chat Template
tokenizer = get_chat_template(tokenizer, chat_template = "qwen-2.5")

# Standardize ShareGPT roles
try:
    dataset = standardize_sharegpt(dataset)
except Exception:
    pass

# Auto-detect conversation columns and format to 'text'
chat_col = None
if "conversations" in dataset.column_names:
    chat_col = "conversations"
elif "messages" in dataset.column_names:
    chat_col = "messages"

if chat_col:
    def format_chat(examples):
        texts = [
            tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
            for convo in examples[chat_col]
        ]
        return {"text": texts}
    dataset = dataset.map(format_chat, batched=True)

elif "instruction" in dataset.column_names:
    alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""
    def format_alpaca(examples):
        instructions = examples["instruction"]
        inputs = examples.get("input", [""] * len(instructions))
        outputs = examples.get("output", [""] * len(instructions))
        texts = [alpaca_prompt.format(inst, inp, out) + tokenizer.eos_token for inst, inp, out in zip(instructions, inputs, outputs)]
        return {"text": texts}
    dataset = dataset.map(format_alpaca, batched=True)

if "text" not in dataset.column_names:
    raise RuntimeError(f"❌ Formatting failed. Columns available: {dataset.column_names}")

print("✅ Dataset formatted successfully into 'text' column!")


# ------------------------------------------------------------------------------
# STEP 8: CONFIGURE LORA & TRAIN 🚀
# ------------------------------------------------------------------------------
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRADIENT_ACCUMULATION_STEPS,
        warmup_steps = WARMUP_STEPS,
        max_steps = MAX_STEPS,
        learning_rate = LEARNING_RATE,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = DRIVE_CHECKPOINT_DIR,
        save_strategy = "steps",
        save_steps = SAVE_STEPS,
        report_to = "none",
    ),
)

print("\n🚀 Starting Model Fine-Tuning...")
trainer_stats = trainer.train()

# Save LoRA adapters directly to Google Drive (Lightweight! ~150MB)
model.save_pretrained(DRIVE_CHECKPOINT_DIR)
tokenizer.save_pretrained(DRIVE_CHECKPOINT_DIR)
print(f"✅ LoRA Adapters saved to Google Drive: {DRIVE_CHECKPOINT_DIR}")


# ------------------------------------------------------------------------------
# STEP 9: SAVE MERGED 16-BIT MODEL LOCALLY 💾
# ------------------------------------------------------------------------------
print(f"\n⚡ Merging & Saving 16-bit model to local Colab storage...")
model.save_pretrained_merged(LOCAL_FINAL_SAVE_DIR, tokenizer, save_method = "merged_16bit")

print(
    f"\n🎉 ALL STEPS COMPLETED SUCCESSFULLY!\n"
    f"📍 LoRA Checkpoints (Drive): {DRIVE_CHECKPOINT_DIR}\n"
    f"📍 Merged Model (Local Colab): {LOCAL_FINAL_SAVE_DIR}\n"
)

In [ ]:
# ==============================================================================
# LIGHTWEIGHT HF UPLOADER (NO UNSLOTH NEEDED! 🚀)
# ==============================================================================
!pip install --quiet huggingface_hub

from huggingface_hub import HfApi
import os

# 1. CONFIGURATION PANEL 🎛️
HF_WRITE_TOKEN = "hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"  #@param {type:"string"}
HF_REPO_NAME = "your-username/my-finetuned-model"        #@param {type:"string"}
FOLDER_PATH = "/content/merged_16bit_model"             #@param {type:"string"}
REPO_VISIBILITY = "public"                             #@param ["private", "public"]

# 2. VERIFY FOLDER EXISTS
if not os.path.exists(FOLDER_PATH):
    raise FileNotFoundError(f"❌ Cannot find folder at: {FOLDER_PATH}")

# 3. INITIALIZE HF API & CREATE REPO
api = HfApi(token=HF_WRITE_TOKEN)

print(f"📁 Preparing Hugging Face repository '{HF_REPO_NAME}' ({REPO_VISIBILITY.upper()})...")
api.create_repo(
    repo_id=HF_REPO_NAME,
    private=(REPO_VISIBILITY == "private"),
    exist_ok=True,
    repo_type="model"
)

# 4. PUSH FOLDER TO HUGGING FACE
print(f"🚀 Uploading all files from '{FOLDER_PATH}'...")
api.upload_folder(
    folder_path=FOLDER_PATH,
    repo_id=HF_REPO_NAME,
    repo_type="model",
)

print(f"\n🎉 BOOM! Your model is live at: https://huggingface.co/{HF_REPO_NAME}")

In [ ]:
# @title ⬇️ 3. Auto-Download Model
# @markdown Pushes your zip file straight into your browser download queue! 🚀

import os
from google.colab import files

zip_file = "/content/merged_16bit_model" #@param {type:"string"}

if os.path.exists(zip_file):
  print("⬇️ Triggering browser download now...")
  files.download(zip_file)
else:
  print(
      "❌ Zip file missing! Make sure to hit play on Cell 3 (Create Zip"
      " Archive) first."
  )